# Obstacles Showcase — Super-Ellipse Gallery

Paper-facing skeleton that produces Fig. 3 (obstacle gallery) for the NeurIPS 2026 submission. The developer walkthrough at `../../notebooks/obstacles.ipynb` is the parameter-exploration surface and stays untouched by paper work; this notebook is the paper-facing export.

Source of truth for paper structure: `../../../passage-paper/neurips/skeleton.md` (v3).

## Paper Mapping

| Claim | `\evidence{}` tag | Output artifact |
| --- | --- | --- |
| Sec. 4.4 obstacle generator produces the documented super-ellipse family | `passage/paper/notebooks/obstacles_showcase.ipynb -> passage/paper/figures/obstacles_showcase.pdf` | `figures/obstacles_showcase.pdf` |

## Inputs / Outputs

| Kind | Path |
| --- | --- |
| Input | `passage.obstacles` super-ellipse generator |
| Input | Fixed list of obstacle-config seeds (documented in the setup cell) |
| Input | Reference elevation tile (real tile if the data share is mounted, otherwise a stub grid) |
| Output | `../figures/obstacles_showcase.pdf` (+ `.png`) |
| Output | `../figures/obstacles_showcase.manifest.json` |


In [ ]:
# Setup: import helpers and handle optional dependencies.
from __future__ import annotations

import sys
from pathlib import Path

PAPER_DIR = Path.cwd().resolve().parent
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

try:
    from helpers import FIGURES_DIR, emit_manifest, save_figure
except ImportError as exc:  # pragma: no cover
    raise ImportError(
        "Could not import passage.paper.helpers; run from passage/paper/notebooks/."
    ) from exc

try:
    import numpy as np
    import matplotlib.pyplot as plt
    HAS_PLOTTING = True
except ImportError:
    HAS_PLOTTING = False

try:
    from passage import obstacles as _obstacles
    HAS_OBSTACLES = True
except ImportError:
    HAS_OBSTACLES = False

STUDY_ID = "obstacles_showcase"

# Gallery parameters — per-obstacle super-ellipse seeds and shape exponents
# spanning the documented log-uniform range on [1, 80]. Rows show rising
# exponent; columns show rising rotation.
GALLERY = [
    {"seed": 42, "exponent": 2.0, "rotation_deg": 0.0},
    {"seed": 42, "exponent": 5.0, "rotation_deg": 30.0},
    {"seed": 42, "exponent": 20.0, "rotation_deg": 60.0},
    {"seed": 42, "exponent": 80.0, "rotation_deg": 90.0},
]
GRID_SIZE = 256
COVERAGE_CAP = 0.30

In [ ]:
# Build the obstacle gallery figure (Fig. 3).
#
# TODO: replace the placeholder binary masks with real calls into
# passage.obstacles once the paper-facing entry point is exposed
# (see passage/src/passage/obstacles.py). The developer walkthrough at
# ../../notebooks/obstacles.ipynb carries the full parameter sweep.

if HAS_PLOTTING:
    fig, axes = plt.subplots(
        1, len(GALLERY), figsize=(3.0 * len(GALLERY), 3.0), squeeze=False
    )
    for ci, params in enumerate(GALLERY):
        rng = np.random.default_rng(seed=params["seed"] + ci)
        stub_mask = (rng.random((GRID_SIZE, GRID_SIZE)) < 0.05).astype(float)
        ax = axes[0, ci]
        ax.imshow(stub_mask, cmap="gray_r")
        ax.set_title(
            f"n={params['exponent']:.0f}, rot={params['rotation_deg']:.0f} deg",
            fontsize=9,
        )
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(
        "PASSAGE super-ellipse obstacle gallery (stub data — replace with real generator)"
    )
    fig.tight_layout()
else:
    fig = None

In [ ]:
# Emit manifest and save figure.
outputs = []
if fig is not None:
    pdf_path = save_figure(fig, study_id=STUDY_ID)
    outputs.extend([pdf_path, pdf_path.with_suffix(".png")])

manifest_path = emit_manifest(
    study_id=STUDY_ID,
    inputs={
        "gallery": GALLERY,
        "grid_size": GRID_SIZE,
        "coverage_cap": COVERAGE_CAP,
        "has_plotting": HAS_PLOTTING,
        "has_obstacles_module": HAS_OBSTACLES,
        "paper_section": "4.4",
        "paper_anchors": ["fig:obstacles"],
    },
    outputs=outputs,
)
print(f"wrote {manifest_path}")